# 01 — EDA: CICIDS2017 Dataset Exploration

> **Mục tiêu:** Hiểu cấu trúc dataset, phân tích phân bố labels, tìm exfiltration proxy patterns

---

## 1. Dataset Overview

Dataset chính: **CICIDS2017_ML-CVE** — 8 files CSV, tổng cộng ~2.6 triệu flows

| File | Rows | Main Labels |
|---|---|---|
| Monday-WorkingHours | 529,918 | BENIGN (100%) |
| Tuesday-WorkingHours | 445,909 | BENIGN (96.9%), FTP-Patator (1.8%), SSH-Patator (1.3%) |
| Wednesday-WorkingHours | 692,703 | BENIGN (63.5%), DoS Hulk (33.4%), DoS variants |
| Thursday-Morning-WebAttacks | 170,366 | BENIGN (98.7%), Web Attacks |
| Thursday-Afternoon-Infilteration | 288,602 | BENIGN (99.99%), **Infiltration (36 flows)** |
| Friday-Morning | 191,033 | BENIGN (99%), **Bot (1,966)** |
| Friday-Afternoon-PortScan | 286,467 | PortScan (55.5%), BENIGN |
| Friday-Afternoon-DDoS | 225,745 | DDoS (56.7%), BENIGN |

## 2. Column Analysis

**79 columns** từ CICFlowMeter. Key observations:

- Columns có space prefix không đều → cần `.str.strip()` khi load
- `Flow Bytes/s` và `Flow Packets/s` chứa **inf values** (cần xử lý)
- `Fwd Header Length` và `min_seg_size_forward` có **negative values** (lỗi CICFlowMeter)
- `Label` là column duy nhất không phải numeric
- Không có null values trong dataset

## 3. Exfiltration Proxy Analysis

⚠️ **CICIDS2017 không có label exfiltration rõ ràng.** Phân tích 3 potential proxies:

### 3.1 Infiltration (Thursday-Afternoon) — Chỉ 36 flows, không đủ

Không đủ samples để train model. Không dùng được.

### 3.2 Bot Traffic (Friday-Morning) — ✅ **EXCELLENT EXFIL PROXY**

**Phát hiện quan trọng nhất:** Bot traffic có signature exfiltration rõ ràng:

| Metric | Bot | BENIGN | Ratio |
|---|---|---|---|
| Total Fwd Bytes (upload) | 2,645 | 579 | **4.57x** |
| Total Bwd Bytes (download) | 64 | 28,680 | **0.002x** |
| Flow Duration | 351s | 11,762s | **0.03x** |
| Flow Bytes/s | 307,099 | 1,786,234 | **0.17x** |
| PSH Flag Count | 0.62 | 0.22 | **2.81x** |
| Average Packet Size | 55 | 121 | **0.46x** |

→ Bot traffic: **upload cao gấp 4.5x** so với download, **session cực ngắn**, **automated behavior** (PSH flags cao) = perfect exfiltration signature

### 3.3 Port Distribution

| Port | Monday (Normal) | Infiltration | Note |
|---|---|---|---|
| 53 (DNS) | 40.5% | 34.4% | Baseline DNS |
| 443 (HTTPS) | 26.6% | 17.9% | HTTPS traffic |
| 80 (HTTP) | 9.6% | 9.0% | HTTP traffic |

Port 22 (SSH) có mặt trong cả Normal và Infiltration. SSH tunneling là potential exfil vector.

## 4. Key Differentiating Features (Normal vs Infiltration)

Features có difference ratio > 2x hoặc < 0.5x:

| Feature | Normal Mean | Infiltration Mean | Ratio | Interpretation |
|---|---|---|---|---|
| `min_seg_size_forward` | -3,614 | 26 | 0.01x | Abnormal segmentation |
| `Fwd Header Length` | -13,204 | 162 | 0.01x | CICFlowMeter artifact |
| `Bwd Header Length` | -3,056 | 161 | 0.05x | CICFlowMeter artifact |
| `Total Length of Bwd Packets` | 17,898 | 6,162 | 0.34x | Less response data |
| `Flow IAT Min` | 293,691 | 74,518 | 0.25x | Faster inter-arrival |
| `Fwd IAT Min` | 1,466,088 | 711,601 | 0.49x | Faster forward packets |
| `Subflow Bwd Bytes` | 17,898 | 6,162 | 0.34x | Less backward traffic |

→ Features **Flow IAT Min** và **Total Length of Bwd Packets** là indicators tốt nhất

## 5. Decision: Exfiltration Labeling Strategy

**Kết luận từ EDA:**

1. **Primary exfil proxy: Bot traffic** (1,966 flows) — signature rõ ràng
2. **Secondary: Infiltration** (36 flows) — quá ít, không train được nhưng dùng để validate
3. **Custom heuristics** để augment dataset:
   - `upload_ratio > 3.0` (fwd_bytes / bwd_bytes > 3)
   - `flow_duration < 600s` (session ngắn)
   - `psh_flag_ratio > 0.5` (automated behavior)
   - `dst_port NOT IN [80, 443, 53, 22]` (unusual port)

**Labeling scheme:**
- Label 0 = Normal traffic (tất cả BENIGN flows)
- Label 1 = Exfiltration (Bot + Infiltration + custom heuristics)

## 6. Data Quality Issues

### 6.1 Infinity Values

`Flow Bytes/s` và `Flow Packets/s` chứa inf khi flow duration = 0.
→ Xử lý: replace inf bằng max finite value hoặc 0

### 6.2 Negative Values

`Fwd Header Length`, `Bwd Header Length`, `min_seg_size_forward` có giá trị âm (CICFlowMeter bug).
→ Xử lý: clip về 0 hoặc bỏ qua các features này

### 6.3 Class Imbalance

| Class | Approximate Count | % |
|---|---|---|
| Normal | ~2,200,000 | ~85% |
| Exfil (Bot) | ~1,966 | ~0.08% |
| DoS/DDoS/PortScan | ~700,000 | ~27% |

→ Supervised model cần **oversampling** hoặc **class_weight** để xử lý imbalance

## 7. Next Steps

1. **Feature Engineering** (`02_Feature_Engineering.ipynb`):
   - Tính custom features: upload_ratio, burst_count, burst_ratio
   - Áp dụng labeling strategy đã chọn
   - Train/test split (70/15/15, stratified)

2. **Pipeline Development** (`src/`):
   - Implement multi-threaded capture → feature → inference pipeline
   - Train models
   - Evaluate